In [30]:
from geometry_msgs.msg import Twist

class Robot:
    def __init__(self, wheel_radius=0.06, robot_radius=0.2):
        """
        Initialize the Robot with given wheel radius and robot radius.
        
        :param wheel_radius: Radius of the wheels
        :param robot_radius: Radius of the robot
        """
        self.wheel_radius = wheel_radius
        self.robot_radius = robot_radius

    def forward_kinematics(self, w_l, w_r):
        """
        Compute the forward kinematics for a differential drive robot.
        
        :param w_l: Angular velocity of the left wheel
        :param w_r: Angular velocity of the right wheel
        :return: Linear velocity (v) and angular velocity (a) of the robot
        """
        c_l = self.wheel_radius * w_l
        c_r = self.wheel_radius * w_r
        v = (c_l + c_r) / 2
        a = (c_r - c_l) / (2 * self.robot_radius)
        return (v, a)

    def inverse_kinematics(self, v, a):
        """
        Compute the inverse kinematics for a differential drive robot.
        
        :param v: Linear velocity of the robot
        :param a: Angular velocity of the robot
        :return: Angular velocities of the left wheel (w_l) and right wheel (w_r)
        """
        c_l = v - (self.robot_radius * a)
        c_r = v + (self.robot_radius * a)
        w_l = c_l / self.wheel_radius
        w_r = c_r / self.wheel_radius
        return (w_l, w_r)

    def inverse_kinematics_from_twist(self, t: Twist):
        """
        Compute the inverse kinematics from a Twist message.
        
        :param t: Twist message containing linear and angular velocities
        :return: Angular velocities of the left wheel (w_l) and right wheel (w_r)
        """
        return self.inverse_kinematics(t.linear.x, t.angular.z)


# Use Inverse Kinemantics to compute wheel velocities, given geometry of the robot

In [31]:

r = Robot(
    wheel_radius=0.033,
    robot_radius=0.1
)

v = 0.0
a = 1.0
(w_l, w_r) = r.inverse_kinematics(v, a)
print(f"given v={v} and a={a}:\n\tw_l = {w_l},\tw_r = {w_r}")


v = 1.0
a = 0.0
(w_l, w_r) = r.inverse_kinematics(v, a)
print(f"given v={v} and a={a}:\n\tw_l = {w_l},\tw_r = {w_r}")



given v=0.0 and a=1.0:
	w_l = -3.0303030303030303,	w_r = 3.0303030303030303
given v=1.0 and a=0.0:
	w_l = 30.3030303030303,	w_r = 30.3030303030303


# Use the forwards kinematics

Now use forward kinematics to estimate the movement of the vehicle, given wheel speeds, e.g. from encoders

In [32]:


(v, a) = r.forward_kinematics(w_l, w_r)
print(f"given wheels speeds w_l={w_l} and w_r={w_r}\n\tv = {v},\ta = {a}")


given wheels speeds w_l=30.3030303030303 and w_r=30.3030303030303
	v = 1.0,	a = 0.0


# Interaction with `Twist`

Assuming we received a `Twist` message, e.g. as part of the information from the `/odom` topic, we can calculate the speeds fo the wheels

In [33]:

from geometry_msgs.msg import Twist
from nav_msgs.msg import Odometry


odom = Odometry()

# fake the odometry (check `ros2 interface show nav_msgs/msg/Odometry` 
# to learn more about the Odometry message type):
t = odom.twist.twist
t.linear.x = 0.3
t.angular.z = 0.8
display(odom)

(w_l, w_r) = r.inverse_kinematics_from_twist(t)
print("given some Twist message:\n\tw_l = %f,\tw_r = %f" % (w_l, w_r))


nav_msgs.msg.Odometry(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=0, nanosec=0), frame_id=''), child_frame_id='', pose=geometry_msgs.msg.PoseWithCovariance(pose=geometry_msgs.msg.Pose(position=geometry_msgs.msg.Point(x=0.0, y=0.0, z=0.0), orientation=geometry_msgs.msg.Quaternion(x=0.0, y=0.0, z=0.0, w=1.0)), covariance=array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0.])), twist=geometry_msgs.msg.TwistWithCovariance(twist=geometry_msgs.msg.Twist(linear=geometry_msgs.msg.Vector3(x=0.3, y=0.0, z=0.0), angular=geometry_msgs.msg.Vector3(x=0.0, y=0.0, z=0.8)), covariance=array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0.])))

given some Twist message:
	w_l = 6.666667,	w_r = 11.515152
